# Interactive Demo: End-to-End Multimodal Inference with GNN-BERT on Real Audio
### Course: Neural Networks (CSE425 / EEE474 / CSE715)
**Project**: GNN-Based BERT for Understanding Context from Music

This demo provides an end-to-end walkthrough on **real music clips** from the MusicCaps dataset:
1. **Input**: A genuine audio recording from `/Users/tanishaislam/MusicCaps/audio/` + its official human expert caption.
2. **Acoustic Graph Builder**: Real audio $\to$ 128-bin mel-spectrogram $\to$ segment structure graph $G = (V, E)$ using temporal adjacency + cosine similarity ($\tau = 0.55$).
3. **Text Tokenizer**: HuggingFace DistilBERT contextual embeddings $H_{\text{text}}$ and $t_{\text{CLS}}$.
4. **Cross-Attention Multimodal Fusion**: $z = [g; A H_{\text{text}}]$ where $A = \text{softmax}(QK^T / \sqrt{d})$.
5. **Context Prediction**: Multi-label tag probabilities, continuous Valence & Arousal emotion coordinates, and token-level cross-modal attention heatmap.

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.audio_features import load_and_resample_audio, extract_segment_features, extract_log_mel_spectrogram
from src.graph_builder import build_segment_graph, to_networkx_graph
from src.bert_encoder import MusicBertEncoder
from src.fusion_model import GNNBertFusionModel
from src.prepare_real_dataset import TOP_TAGS as TAGS_VOCAB

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f"Running Demo Inference on Device: {device}")

## 1. Load Multimodal Fusion Model & BERT Tokenizer

In [ ]:
model = GNNBertFusionModel(
    num_tags=len(TAGS_VOCAB),
    in_node_dim=32,
    gnn_hidden_dim=128,
    gnn_out_dim=128,
    fusion_type='cross_attention'
).to(device)

bert_tok = MusicBertEncoder()
model.eval()
print("Multimodal GNN-BERT Fusion Model successfully initialized!")

## 2. Load Real Audio Clip & Human Expert Caption (MusicCaps)

In [ ]:
audio_path = "/Users/tanishaislam/MusicCaps/audio/mc_001.wav"
if not os.path.exists(audio_path):
    audio_path = "/Users/tanishaislam/MusicCaps/MagnaTagATune/audio/aba_structure-epic-01-deep_step-117-146.mp3"

test_audio = load_and_resample_audio(audio_path, target_sr=22050, duration=10.0)
test_caption = "The low quality recording features a ballad song that contains sustained strings, mellow piano melody and soft female vocal singing over it."

print("Loaded Real Audio:", audio_path)
print("Real Expert Caption:", test_caption)

## 3. Extract Real Segment Graph & Run End-to-End Inference

In [ ]:
# 1. Real Audio -> Segment features -> Structure Graph
seg_feats = extract_segment_features(test_audio, sr=22050, segment_duration=2.5)
g_dict = build_segment_graph(seg_feats, similarity_threshold=0.55)

x_tensor = torch.tensor(g_dict['x'], dtype=torch.float32).to(device)
edge_index = torch.tensor(g_dict['edge_index'], dtype=torch.long).to(device)
edge_weight = torch.tensor(g_dict['edge_weight'], dtype=torch.float32).to(device)

# 2. Real Caption -> BERT Tokenizer
toks = bert_tok.tokenize([test_caption], device=device)
input_ids = toks['input_ids']
att_mask = toks.get('attention_mask')

# 3. Multimodal Forward Pass
with torch.no_grad():
    out = model(x_tensor, edge_index, input_ids, attention_mask=att_mask, edge_weight=edge_weight)

tag_probs = out['tag_probs'].squeeze(0).cpu().numpy()
valence, arousal = out['emotion_preds'].squeeze(0).cpu().numpy()
att_matrix = out['att_weights'].squeeze(0).squeeze(0).cpu().numpy()

print("=== Inference Results on Real Clip ===")
print(f"Predicted Valence: {valence:.2f} / 9.0")
print(f"Predicted Arousal: {arousal:.2f} / 9.0")

top_indices = np.argsort(-tag_probs)[:5]
print("\nTop-5 Predicted Music Context Tags:")
for rank, idx in enumerate(top_indices, 1):
    print(f"  {rank}. {TAGS_VOCAB[idx]:<14} | Probability: {tag_probs[idx]:.4f}")

## 4. Visualizing Cross-Attention Alignment on Real Caption Words

In [ ]:
words = test_caption.split()[:12]
att_sub = att_matrix[:len(words)].reshape(1, -1)

plt.figure(figsize=(10, 3.5))
sns.heatmap(
    att_sub,
    xticklabels=words,
    yticklabels=['Audio Graph (g)'],
    cmap='Blues',
    annot=True,
    fmt='.2f',
    cbar_kws={'label': 'Attention Weight'}
)
plt.title('Cross-Attention: Audio Structure Graph g attending to Real Caption Words', fontsize=12, fontweight='bold')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()